In [1]:
# ================================
# 1. Install Dependencies
# ================================
!pip install unsloth transformers datasets scikit-learn accelerate

# ================================
# 2. Imports
# ================================
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from tqdm import tqdm

# For auto download (Colab)
from google.colab import files

# ================================
# 3. Load Your Dataset
# ================================
file_path = "/content/OS_relations_fixed.json"

with open(file_path, "r") as f:
    data = json.load(f)

print("Total samples:", len(data))

dataset = Dataset.from_list(data)

# ================================
# 4. Load Model
# ================================
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ================================
# 5. Define Zero-Shot Prompt
# ================================
def format_prompt(example):
    return f"""### Instruction:
{example['instruction']}

Choose ONLY from:
used_for, based_on, implements, part_of, improves, no_relation

### Input:
{example['input']}

### Response:
"""

# ================================
# 6. Clean Prediction (IMPORTANT)
# ================================
VALID_LABELS = ["used_for", "based_on", "implements", "part_of", "improves", "no_relation"]

def clean_prediction(pred):
    pred = pred.lower()
    for label in VALID_LABELS:
        if label in pred:
            return label
    return "no_relation"

# ================================
# 7. Inference Function
# ================================
def predict(example):
    prompt = format_prompt(example)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.0,
            do_sample=False
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    pred_raw = response.split("### Response:")[-1].strip().split("\n")[0]
    pred = clean_prediction(pred_raw)

    return pred

# ================================
# 8. Run Predictions + Store Triplets
# ================================
y_true = []
y_pred = []
results = []

for example in tqdm(dataset):
    text = example["input"]
    gt = example["output"].strip()
    pred = predict(example)

    y_true.append(gt)
    y_pred.append(pred)

    # Store triplets
    results.append({
        "input": text,
        "ground_truth": gt,
        "predicted": pred
    })

# ================================
# 9. Save JSON File
# ================================
output_file = "/content/zero_shot_triplets.json"

with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\n✅ Triplets saved to: {output_file}")

# Auto-download
files.download(output_file)

# ================================
# 10. Evaluation Metrics
# ================================
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\n===== ZERO-SHOT RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n===== Classification Report =====")
print(classification_report(y_true, y_pred))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y_true, y_pred))

# ================================
# 11. Show Sample Predictions
# ================================
for i in range(min(5, len(dataset))):
    print("\n--- Example ---")
    print("Input :", dataset[i]["input"])
    print("GT    :", y_true[i])
    print("Pred  :", y_pred[i])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...

  0%|          | 0/1957 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=10) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 1957/1957 [58:09<00:00,  1.78s/it]


✅ Triplets saved to: /content/zero_shot_triplets.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


===== ZERO-SHOT RESULTS =====
Accuracy : 0.1886
Precision: 0.5976
Recall   : 0.1886
F1 Score : 0.1478

===== Classification Report =====
              precision    recall  f1-score   support

    based_on       0.50      0.06      0.10        70
  implements       0.06      0.03      0.04        99
    improves       0.09      0.18      0.12        34
 no_relation       0.70      0.11      0.18       915
     part_of       0.79      0.03      0.07       558
    used_for       0.14      0.85      0.25       281

    accuracy                           0.19      1957
   macro avg       0.38      0.21      0.13      1957
weighted avg       0.60      0.19      0.15      1957


===== Confusion Matrix =====
[[  4   2   0   4   0  60]
 [  0   3   0   0   0  96]
 [  0   0   6   1   0  27]
 [  1  20  34  97   5 758]
 [  1   8  15  26  19 489]
 [  2  15  14  10   0 240]]

--- Example ---
Input : Sentence: FAT (File Allocation Table): An older file system used by older versions of Windows and oth

In [ ]:
!rm -rf /content/unsloth_compiled_cache